# Exercise 3 — build_status and mark_phase_done

`build_status` gives a snapshot of progress: how many phases are done, the completion percentage, and whether the project is fully complete. `mark_phase_done` mutates a `Phase` in place — it is the single function that drives the tracker forward. Together they form the minimal state machine for a project tracker.

In [ ]:
import datetime
from dataclasses import dataclass, field

@dataclass
class CapstoneSpec:
    name: str; tagline: str; domain: str; description: str
    sections_used: list; deliverables: list; tech_stack: list

@dataclass
class Phase:
    name: str; tasks: list; done: bool = False

@dataclass
class CapstoneReport:
    spec: CapstoneSpec
    phases: list = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.date.today().isoformat())
    completed_at: str = ""

_SPEC = CapstoneSpec(
    name          = "AI Trading Bot",
    tagline       = "Paper-trading bot with sentiment and technical signals.",
    domain        = "finance",
    description   = "End-to-end AI trading bot that fetches OHLCV data, computes "
                    "technical indicators, scores news sentiment with an LLM, applies "
                    "risk controls, and runs a daily paper-trading loop with logging.",
    sections_used = [
        "Section 3: Data & Analysis (pandas, SQLite)",
        "Section 4: Real Apps (FastAPI endpoint)",
        "Section 6: AI Agents (scheduling loop)",
        "Section 7: Finance & Trading (backtester, risk manager, paper trader)",
    ],
    deliverables  = [
        "paper_trader.py with buy/sell/portfolio_value",
        "bot_runner.py with daily scheduling and logging",
        "risk.py with stop-loss and drawdown controls",
        "Deployed FastAPI endpoint",
        "Portfolio case study",
    ],
    tech_stack    = ["Python", "pandas", "Ollama", "SQLite", "FastAPI"],
)
def validate_spec(spec):
    errors = []
    for field_name in ("name", "tagline", "domain", "description"):
        val = getattr(spec, field_name, "")
        if not isinstance(val, str) or not val.strip():
            errors.append(f"{field_name} must be a non-empty string")
    if len(spec.deliverables) < 3:
        errors.append(f"at least 3 deliverables required, got {len(spec.deliverables)}")
    if len(spec.tech_stack) < 2:
        errors.append(f"at least 2 tech stack items required, got {len(spec.tech_stack)}")
    if not spec.sections_used:
        errors.append("sections_used must reference at least one course section")
    return len(errors) == 0, errors
def generate_project_plan(spec):
    deliverable_list = "\n".join(f"- [ ] {d}" for d in spec.deliverables)
    tech_list        = "\n".join(f"- {t}"     for t in spec.tech_stack)
    sections_list    = "\n".join(f"- {s}"     for s in spec.sections_used)
    return (
        f"# {spec.name} — Implementation Plan\n\n"
        f"**{spec.tagline}**\n\n"
        f"## Overview\n\n{spec.description}\n\n"
        f"## Course Sections Applied\n\n{sections_list}\n\n"
        f"## Tech Stack\n\n{tech_list}\n\n"
        f"## Deliverables\n\n{deliverable_list}\n\n"
        f"## Build Phases\n\n"
        f"### Phase 1 — Plan\n- [ ] Finalise spec\n- [ ] Write gate tests\n\n"
        f"### Phase 2 — Build\n- [ ] Implement core AI\n- [ ] Wire pipeline\n\n"
        f"### Phase 3 — Test\n- [ ] Gate green\n- [ ] End-to-end test\n\n"
        f"### Phase 4 — Deploy\n- [ ] Write .env\n- [ ] Deploy\n\n"
        f"### Phase 5 — Document\n- [ ] README\n- [ ] Case study\n\n"
        f"### Phase 6 — Share\n- [ ] GitHub\n- [ ] Portfolio\n- [ ] Post\n"
    )
def default_phases(spec):
    ai_backend = next(
        (t for t in spec.tech_stack if t.lower() in ("ollama","llama","llamacpp")),
        "AI backend",
    )
    return [
        Phase("Plan",  [f"Finalise spec for {spec.name}", "Install packages", "Write gate tests"]),
        Phase("Build", ["Implement core AI", f"Wire {ai_backend}", "Build pipeline"]),
        Phase("Test",  ["Gate: all checks green", "Happy path", "Edge cases"]),
        Phase("Deploy",["Write .env", "Deploy to production", "Verify live URL"]),
        Phase("Document",["README", "Case study", "Demo video"]),
        Phase("Share", ["Push to GitHub", "Portfolio", "Post on LinkedIn"]),
    ]

def build_status(report):
    """Return build phase completion summary.

    Returns:
        dict with keys:
          phases           : list[dict] — {name, done, n_tasks}
          total_phases     : int
          completed_phases : int
          completion_pct   : float  (0.0–100.0, 1 decimal place)
          is_complete      : bool   (all phases done and total > 0)
    """
    total = len(report.phases)
    done  = sum(1 for p in report.phases if p.done)
    # TODO: return the status dict
    return {}


def mark_phase_done(report, phase_name):
    """Mark the named phase done (case-sensitive).

    Returns True if found and marked; False if no phase with that name.
    """
    # TODO: iterate report.phases; set done=True and return True if found
    return False


### Checks

In [ ]:
checks = 0

report = CapstoneReport(spec=_SPEC, phases=default_phases(_SPEC))

# 1 — build_status on fresh report: 0 completed, 0%
try:
    status = build_status(report)
    assert status["total_phases"]     == 6,   f"expected 6, got {status['total_phases']}"
    assert status["completed_phases"] == 0,   f"expected 0, got {status['completed_phases']}"
    assert status["completion_pct"]   == 0.0, f"expected 0.0, got {status['completion_pct']}"
    assert status["is_complete"]      is False
    checks += 1; print("✅ 1 fresh report: 6 phases, 0 done, 0.0%, not complete")
except Exception as e:
    print("❌ 1:", e)

# 2 — mark_phase_done: True for valid name, False for invalid
try:
    r2 = mark_phase_done(report, "Plan")
    r3 = mark_phase_done(report, "NoSuchPhase")
    assert r2 is True,  f"expected True for 'Plan', got {r2}"
    assert r3 is False, f"expected False for 'NoSuchPhase', got {r3}"
    checks += 1; print("✅ 2 mark_phase_done: True for valid name, False for invalid")
except Exception as e:
    print("❌ 2:", e)

# 3 — after marking Plan done: 1 completed, ~16.7%
try:
    status = build_status(report)
    assert status["completed_phases"] == 1
    assert abs(status["completion_pct"] - 16.7) < 0.1,         f"expected ~16.7%, got {status['completion_pct']}"
    assert status["is_complete"] is False
    checks += 1; print(f"✅ 3 after Plan done: 1/6 phases, {status['completion_pct']}%")
except Exception as e:
    print("❌ 3:", e)

# 4 — mark all 6 phases done → is_complete=True, 100%
try:
    for phase_name in ["Build", "Test", "Deploy", "Document", "Share"]:
        mark_phase_done(report, phase_name)
    status = build_status(report)
    assert status["completed_phases"] == 6
    assert status["completion_pct"]   == 100.0
    assert status["is_complete"]      is True
    checks += 1; print("✅ 4 all 6 phases done: is_complete=True, 100%")
except Exception as e:
    print("❌ 4:", e)

# 5 — phases list in status has correct structure
try:
    status = build_status(report)
    assert len(status["phases"]) == 6
    for p in status["phases"]:
        assert {"name","done","n_tasks"}.issubset(p.keys())
        assert isinstance(p["n_tasks"], int) and p["n_tasks"] == 3
    checks += 1; print("✅ 5 each phase dict has name, done, n_tasks=3")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
